# Proyecto Integrador ADBD 2026 — Fase 1
## NYC 311: predicción temprana del riesgo de atención prolongada

Notebook reproducible con datos reales de solicitudes ciudadanas de NYC 311 (2021–2024). La unidad de análisis es **una solicitud de servicio** (`unique_key`). El objetivo futuro será estimar, al momento del ingreso, la probabilidad de que una solicitud tarde más que el umbral histórico de su tipo de reclamo. Esta definición evita depender de `due_date`, cuya cobertura es insuficiente, y permite priorizar carga operativa sin usar información del futuro.

Fuentes oficiales: [NYC 311](https://portal.311.nyc.gov/about-nyc-311/), [NYC Open Data](https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2020-to-Present/erm2-nwe9) y [diccionario/metadatos API](https://data.cityofnewyork.us/api/views/erm2-nwe9). Corte descargado el 11-09-2026. El conjunto se actualiza diariamente; por eso los resultados se fijan con los archivos locales de esta entrega.

In [ ]:
from pathlib import Path
import os, time, json, statistics
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F, types as T

ROOT = Path.cwd()
if not (ROOT/'data'/'raw').exists() and (ROOT.parent/'data'/'raw').exists():
    ROOT = ROOT.parent
RAW = ROOT/'data'/'raw'
PARQUET = ROOT/'data'/'silver'/'nyc311_v3'
OUT = ROOT/'outputs_fase1'
EVID = ROOT/'evidencia'
for p in (PARQUET, OUT, EVID): p.mkdir(parents=True, exist_ok=True)

spark = (SparkSession.builder.appName('NYC311-Fase1')
         .master('local[*]').config('spark.driver.memory','6g')
         .config('spark.sql.shuffle.partitions','12')
         .config('spark.sql.adaptive.enabled','true')
         .config('spark.sql.parquet.int96RebaseModeInWrite','CORRECTED').getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Spark', spark.version, '| raíz:', ROOT)

## 1. Ingesta y capa analítica

Se seleccionan solo 18 columnas pertinentes. La copia analítica se escribe en Parquet, particionada por año, para habilitar poda de columnas, filtros por partición y lectura eficiente. Los CSV oficiales permanecen como capa cruda e inmutable.

In [ ]:
schema = T.StructType([
 T.StructField('unique_key',T.StringType()), T.StructField('created_date',T.StringType()),
 T.StructField('closed_date',T.StringType()), T.StructField('agency',T.StringType()),
 T.StructField('agency_name',T.StringType()), T.StructField('complaint_type',T.StringType()),
 T.StructField('descriptor',T.StringType()), T.StructField('location_type',T.StringType()),
 T.StructField('incident_zip',T.StringType()), T.StructField('borough',T.StringType()),
 T.StructField('status',T.StringType()), T.StructField('due_date',T.StringType()),
 T.StructField('resolution_action_updated_date',T.StringType()), T.StructField('community_board',T.StringType()),
 T.StructField('council_district',T.StringType()), T.StructField('open_data_channel_type',T.StringType()),
 T.StructField('latitude',T.DoubleType()), T.StructField('longitude',T.DoubleType())])

if not any(PARQUET.rglob('*.parquet')):
    raw = (spark.read.option('header',True).option('multiLine',True).option('escape','\"')
           .schema(schema).csv([str(RAW/f'nyc311_{y}.csv') for y in range(2021,2025)]))
    curated = (raw
      .withColumn('created_ts',F.to_timestamp('created_date',"yyyy-MM-dd'T'HH:mm:ss.SSS"))
      .withColumn('closed_ts',F.to_timestamp('closed_date',"yyyy-MM-dd'T'HH:mm:ss.SSS"))
      .withColumn('due_ts',F.to_timestamp('due_date',"yyyy-MM-dd'T'HH:mm:ss.SSS"))
      .withColumn('created_year',F.year('created_ts'))
      .withColumn('created_month',F.date_format('created_ts','yyyy-MM'))
      .withColumn('resolution_hours',(F.unix_timestamp('closed_ts')-F.unix_timestamp('created_ts'))/3600)
      .drop('created_date','closed_date','due_date'))
    curated.write.mode('overwrite').partitionBy('created_year').parquet(str(PARQUET))

df = spark.read.parquet(str(PARQUET)).filter(F.col('created_year').between(2021,2024))
df.createOrReplaceTempView('requests')
print('Parquet listo:', PARQUET)

## 2. Calidad, escala y aptitud para modelar

Los controles verifican escala, grano, duplicados, fechas y cobertura. `due_date` se audita explícitamente porque parecía una etiqueta SLA natural; si su cobertura es baja, no se utilizará como objetivo principal.

In [ ]:
quality = spark.sql('''
SELECT count(*) n,
       count(DISTINCT unique_key) unique_keys,
       sum(CASE WHEN created_ts IS NULL THEN 1 ELSE 0 END) null_created,
       sum(CASE WHEN closed_ts IS NOT NULL THEN 1 ELSE 0 END) with_closed,
       sum(CASE WHEN due_ts IS NOT NULL THEN 1 ELSE 0 END) with_due,
       sum(CASE WHEN latitude IS NOT NULL AND longitude IS NOT NULL THEN 1 ELSE 0 END) with_geo,
       sum(CASE WHEN resolution_hours < 0 THEN 1 ELSE 0 END) negative_duration
FROM requests''').toPandas()
quality['closed_pct']=100*quality.with_closed/quality.n
quality['due_pct']=100*quality.with_due/quality.n
quality['geo_pct']=100*quality.with_geo/quality.n
display(quality.T)
assert int(quality.n.iloc[0]) == 13072334, 'Conteo distinto al corte esperado'
assert int(quality.unique_keys.iloc[0]) == int(quality.n.iloc[0]), 'La llave no es única'
quality.to_csv(EVID/'calidad_resumen.csv',index=False)

In [ ]:
annual = spark.sql('''SELECT created_year AS year, count(*) AS requests,
 round(100*avg(CASE WHEN closed_ts IS NOT NULL THEN 1 ELSE 0 END),2) closed_pct,
 percentile_approx(CASE WHEN resolution_hours>=0 THEN resolution_hours END,0.5) median_resolution_h
 FROM requests GROUP BY created_year ORDER BY year''').toPandas()
display(annual)
annual.to_csv(EVID/'conteos_anuales.csv',index=False)

**Decisión de etiqueta.** `due_date` no es suficientemente completo para representar a toda la operación. Para Fase 2 se construirá, únicamente con solicitudes cerradas y duración no negativa, un umbral de atención prolongada por `complaint_type` aprendido en el período de entrenamiento 2021–2023 (por ejemplo, percentil 75). La etiqueta será `resolution_hours > umbral_tipo`. Las variables posteriores al ingreso —cierre, estado final, descripción de resolución y fecha de actualización— quedan excluidas de los predictores para impedir fuga de información. La evaluación será temporal: entrenamiento 2021–2023 y prueba 2024.

## 3. EDA a escala con Spark SQL

Se combinan agregaciones temporales, segmentación territorial y dos funciones de ventana con interpretación operativa.

In [ ]:
top_complaints = spark.sql('''SELECT complaint_type, count(*) requests,
 round(100*count(*)/sum(count(*)) over(),2) share_pct,
 round(percentile_approx(CASE WHEN resolution_hours>=0 THEN resolution_hours END,0.5),1) median_resolution_h
 FROM requests WHERE complaint_type IS NOT NULL
 GROUP BY complaint_type ORDER BY requests DESC LIMIT 12''').toPandas()
display(top_complaints)
top_complaints.to_csv(EVID/'top_complaints.csv',index=False)

In [ ]:
monthly_borough = spark.sql('''
WITH m AS (
 SELECT created_month, borough, count(*) requests
 FROM requests WHERE borough IN ('BRONX','BROOKLYN','MANHATTAN','QUEENS','STATEN ISLAND')
 GROUP BY created_month, borough)
SELECT created_month, borough, requests,
 round(avg(requests) OVER (PARTITION BY borough ORDER BY created_month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW),1) rolling_3m,
 dense_rank() OVER (PARTITION BY created_month ORDER BY requests DESC) demand_rank
FROM m ORDER BY created_month, demand_rank''').toPandas()
display(monthly_borough.tail(10))
monthly_borough.to_csv(EVID/'ventana_demanda_territorial.csv',index=False)

**Ventana 1.** El promedio móvil de tres meses separa tendencia de ruido mensual; el ranking identifica el borough con mayor presión relativa en cada período. Juntos permiten dimensionar capacidad y redistribuir recursos con una señal menos volátil.

In [ ]:
agency_window = spark.sql('''
WITH m AS (
 SELECT created_month, agency, count(*) requests,
        percentile_approx(CASE WHEN resolution_hours>=0 THEN resolution_hours END,0.5) median_h
 FROM requests WHERE agency IS NOT NULL GROUP BY created_month, agency),
w AS (
 SELECT *, lag(median_h) OVER(PARTITION BY agency ORDER BY created_month) prev_median_h
 FROM m)
SELECT created_month, agency, requests, round(median_h,1) median_h,
 round(median_h-prev_median_h,1) mom_delta_h,
 dense_rank() OVER(PARTITION BY created_month ORDER BY median_h DESC) latency_rank
FROM w WHERE requests>=1000 ORDER BY created_month, latency_rank''').toPandas()
display(agency_window.tail(10))
agency_window.to_csv(EVID/'ventana_latencia_agencias.csv',index=False)

**Ventana 2.** `lag` calcula el cambio mensual de la mediana de resolución por agencia, mientras el ranking compara latencia entre agencias con volumen material. Un aumento sostenido de `mom_delta_h` es una alerta temprana de acumulación operativa, no una prueba causal.

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(10,4.8))
for b,g in monthly_borough.groupby('borough'):
    ax.plot(g.created_month,g.rolling_3m,label=b,linewidth=2)
ax.set_xticks(range(0,len(sorted(monthly_borough.created_month.unique())),6))
ax.set_xticklabels(sorted(monthly_borough.created_month.unique())[::6],rotation=45,ha='right')
ax.set_ylabel('Solicitudes (promedio móvil 3 meses)'); ax.set_xlabel('')
ax.set_title('Presión de demanda mensual por borough'); ax.legend(ncol=3,fontsize=8)
fig.tight_layout(); fig.savefig(OUT/'demanda_borough.png',dpi=180); plt.show()

## 4. Benchmark: Spark vs DuckDB vs Polars

Se ejecutan tres consultas equivalentes sobre el mismo Parquet. Cada motor recibe una corrida de calentamiento y tres corridas medidas; se reporta la mediana. Las consultas cubren agregación completa, percentil por categoría y filtro por partición. El benchmark es local y orienta la elección del motor para esta escala; no pretende generalizar a un clúster.

In [ ]:
import sys
import duckdb, polars as pl

glob = str(PARQUET/'**'/'*.parquet')
queries = {
'Q1_agregacion_anual': "SELECT created_year, count(*) n FROM read_parquet(?) GROUP BY 1 ORDER BY 1",
'Q2_categoria': "SELECT complaint_type, count(*) n FROM read_parquet(?) GROUP BY 1 ORDER BY n DESC LIMIT 20",
'Q3_filtro_2024': "SELECT borough, count(*) n FROM read_parquet(?) WHERE created_year=2024 GROUP BY 1 ORDER BY n DESC"}
spark_sql = {
'Q1_agregacion_anual': 'SELECT created_year, count(*) n FROM requests GROUP BY 1 ORDER BY 1',
'Q2_categoria': 'SELECT complaint_type, count(*) n FROM requests GROUP BY 1 ORDER BY n DESC LIMIT 20',
'Q3_filtro_2024': 'SELECT borough, count(*) n FROM requests WHERE created_year=2024 GROUP BY 1 ORDER BY n DESC'}

def bench(fn):
    fn()
    xs=[]
    for _ in range(3):
        t=time.perf_counter(); fn(); xs.append(time.perf_counter()-t)
    return xs

rows=[]; con=duckdb.connect()
for q in queries:
    s=bench(lambda q=q: spark.sql(spark_sql[q]).collect())
    d=bench(lambda q=q: con.execute(queries[q],[glob]).fetchall())
    if q=='Q1_agregacion_anual':
        lf=lambda: pl.scan_parquet(glob,hive_partitioning=True).group_by('created_year').len().sort('created_year').collect()
    elif q=='Q2_categoria':
        lf=lambda: pl.scan_parquet(glob,hive_partitioning=True).group_by('complaint_type').len().sort('len',descending=True).head(20).collect()
    else:
        lf=lambda: pl.scan_parquet(glob,hive_partitioning=True).filter(pl.col('created_year')==2024).group_by('borough').len().sort('len',descending=True).collect()
    p=bench(lf)
    for engine,times in [('Spark',s),('DuckDB',d),('Polars',p)]:
        rows.append({'query':q,'engine':engine,'run1_s':times[0],'run2_s':times[1],'run3_s':times[2],'median_s':statistics.median(times)})
benchmarks=pd.DataFrame(rows)
display(benchmarks.round(3))
benchmarks.to_csv(EVID/'benchmark_3_corridas.csv',index=False)

In [ ]:
# Validación semántica: conteos totales por año coinciden entre motores
scheck = {int(r.created_year):int(r.n) for r in spark.sql(spark_sql['Q1_agregacion_anual']).collect()}
dcheck = dict(con.execute(queries['Q1_agregacion_anual'],[glob]).fetchall())
pcheck = dict(zip(pl.scan_parquet(glob,hive_partitioning=True).group_by('created_year').len().collect()['created_year'].to_list(),
                  pl.scan_parquet(glob,hive_partitioning=True).group_by('created_year').len().collect()['len'].to_list()))
assert scheck == {int(k):int(v) for k,v in dcheck.items()} == {int(k):int(v) for k,v in pcheck.items()}
print('Validación cruzada OK:', scheck)

In [ ]:
fig, ax=plt.subplots(figsize=(9,4.5))
piv=benchmarks.pivot(index='query',columns='engine',values='median_s')
piv.plot(kind='bar',ax=ax,color=['#0B6E99','#E07A2D','#4C956C'])
ax.set_yscale('log'); ax.set_ylabel('Mediana (s, escala log)'); ax.set_xlabel('')
ax.set_title('Benchmark local: mediana de tres corridas'); ax.tick_params(axis='x',rotation=0)
fig.tight_layout(); fig.savefig(OUT/'benchmark_motores.png',dpi=180); plt.show()

### Lectura de planes de ejecución

Spark debe mostrar `FileScan parquet` con `ReadSchema` reducido, `PartitionFilters` en Q3 y `Exchange hashpartitioning` antes de agregaciones. DuckDB y Polars también aplican proyección/filtro en el escaneo. Se guardan los planes para auditoría.

In [ ]:
plans={}
for q,sql in spark_sql.items():
    plans[q]=spark.sql(sql)._jdf.queryExecution().toString()
    (EVID/f'plan_spark_{q}.txt').write_text(plans[q],encoding='utf-8')
print(plans['Q3_filtro_2024'][:5000])

## 5. Recomendación para las fases siguientes

**Motor principal: Spark.** Aunque DuckDB o Polars suelen ser más rápidos en una estación única para consultas exploratorias, Spark es la elección coherente para el pipeline acumulativo: SQL distribuido, MLlib, Structured Streaming y escalamiento horizontal. DuckDB queda como verificador local de bajo costo.

**Arquitectura.** Capa Bronze con CSV oficiales inmutables; Silver en Parquet particionado por año/mes con tipos y reglas de calidad; Gold con features sin fuga, umbrales de latencia y agregados. En Fase 2: pipeline Spark ML + MLflow, split temporal 2021–2023/2024. En Fase 3: reproducción de `created_ts` en Kafka, Structured Streaming con watermark y ventanas temporales.

**Costo estimado.** Para este volumen, el desarrollo local tiene costo incremental de infraestructura cercano a USD 0. En nube, una ejecución batch ocasional en un clúster pequeño de 2–4 workers durante 1–2 horas debiera quedar en el orden de **USD 1–5 por corrida**, sujeto al proveedor, región, almacenamiento y tarifa vigente. Es una estimación paramétrica, no una cotización; se recalibrará con métricas reales de Fase 2.

**Uso de IA generativa.** Se utilizó un asistente para proponer estructura, consultas y código. Cada salida fue validada mediante conteos oficiales, aserciones de llave y escala, comparación entre tres motores, inspección de planes y ejecución completa del notebook. Las interpretaciones son descriptivas y no atribuyen causalidad.

In [ ]:
summary={'rows':int(quality.n.iloc[0]),'unique_keys':int(quality.unique_keys.iloc[0]),
 'closed_pct':float(quality.closed_pct.iloc[0]),'due_pct':float(quality.due_pct.iloc[0]),
 'geo_pct':float(quality.geo_pct.iloc[0]),
 'benchmark':benchmarks.groupby('engine').median(numeric_only=True)['median_s'].round(3).to_dict()}
(EVID/'resumen_ejecutivo.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(json.dumps(summary,indent=2))
spark.stop()